<a href="https://colab.research.google.com/github/saad-muddasser/test_repo/blob/starting_branch/demo/CAM_DS_Retrieval_augumented_generation(RAG)_2_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**First things first** - please go to 'File' and select 'Save a copy in Drive' so that you have your own version of this activity set up and ready to use.
Remember to update the portfolio index link to your own work once completed!

#Demonstration 2.2.1 Retrieval augmented generation
In this demonstration, you will use the LangChain package to perform RAG with LLMs, and learn how to:
- Load, preprocess, and split documents from multiple sources and different formats into manageable chunks.
- Combine document retrieval and LLM generation for context-aware answers.
- Generate embeddings and perform searches to retrieve relevant documents.
- Implement a full RAG pipeline combining a retriever and LLM.
- Use OpenAI models in addition to Hugging Face models.



**Important**: The demonstration uses closed-source models from OpenAI that require API keys. You will be advised to register for an account at the OpenAI developer platform if you do not already have one. The provision of API keys is restricted to personal usage only and is subject to OpenAI’s rate limits. At the time of writing this programme, a sufficient quota of API keys was being offered without charge, but a recent change at OpenAI required that anyone requesting free keys had to add a small credit to their account for the query to work. You will be reimbursed for this credit.

#### Get your OpenAI key

1. Log in at [OpenAI developer platform](https://platform.openai.com/api-keys).
2. Create a new secret key.
3. Copy and paste the key into a document for safe-keeping.
4. In Colab, select the key icon in the left sidebar to open **Secrets**. Add a secret named `OPENAI_API_KEY`, paste the key as its value, and enable notebook access. Do not paste the key into a notebook cell.

Note that each time you run an OpenAI code cell, it sends a request to OpenAI to use the API and may incur usage charges. This demonstration uses GPT-5.6 Luna with low reasoning effort to keep requests small and responsive.


In [ ]:

!pip install -q torch transformers accelerate bitsandbytes transformers sentence-transformers faiss-cpu
!pip install datasets
!pip install langchain-community
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install langchain-text-splitters
!pip install langchain-openai
!pip install rapidocr-onnxruntime
!pip install pypdf
!pip install langchain
!pip install langchain-chroma
!pip install openai


## Load the documents

### Use the CSV loader

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
dataset['train']['context'][0]

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [ ]:
dataset['train']['question'][0]

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'

In [ ]:
dataset['train']['answers'][0]

{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

In [ ]:
dataset['train']['answers'][0]['text']

['Saint Bernadette Soubirous']

In [ ]:
texts = dataset['train']['context']
questions  = dataset['train']['question']
answers  = dataset['train']['answers']

In [ ]:
answers = [ans['text'] for ans in answers]

In [ ]:
import pandas as pd
df = pd.DataFrame()
df['context'] = texts
df['question'] = questions
df['answer'] = answers

In [ ]:
df = df.sample(n=10)

In [ ]:
df

,context,question,answer
58521,"In July 2015, Eton accidentally sent emails to...",For how many students was the email mistake or...,[nine]
75851,"As of 2012[update], there are over 3.5 million...",What percentage of Hyderabad city was covered ...,[9.5%]
5850,"Similar to the other Eur-A countries, most Por...",What are the two main components of cardiovasc...,[ischaemic heart disease and cerebrovascular d...
40964,"In a conventional lamp, the evaporated tungste...",What are the primary causes of light loss?,[Light loss is due to filament evaporation and...
42678,This allowed any English firm to trade with In...,enlish firms were allow to trade with India u...,[prohibited by act of parliament]
56845,"John Calvin supported the ""agent of God"" Chris...",What is the blood of the lamb?,"[had a cleansing nature, similar to baptismal ..."
45648,Early-years education is quite common in Thuri...,When do children graduate from primary school ...,[At the age of ten]
26460,"By 1860, Houston had emerged as a commercial a...",What type of roads converged in Houston?,[rail lines]
79212,Ibn Tufail (Abubacer) and Ibn al-Nafis were pi...,What Arabic book is Ibn Tufail noted for writing?,[Hayy ibn Yaqdhan (Philosophus Autodidactus)]
32695,"On 9 July 2006, during Mass at Valencia's Cath...",What is the name of the cup that some Catholic...,[Santo Caliz]


In [ ]:
df.to_csv("/content/squad.csv", index=False)

In [ ]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv('/content/squad.csv')
data = [
    Document(
        page_content="\n".join(f"{column}: {row[column]}" for column in df.columns),
        metadata={"source": "/content/squad.csv", "row": index},
    )
    for index, row in df.iterrows()
]


In [ ]:
data

[Document(metadata={'source': '/content/squad.csv', 'row': 0}, page_content='context: In July 2015, Eton accidentally sent emails to 400 prospective students, offering them conditional entrance to the school in September 2017. The email was intended for nine students, but an IT glitch caused the email to be sent to 400 additional families, who didn\'t necessarily have a place. In response, the school issued the following statement: "This error was discovered within minutes and each family was immediately contacted to notify them that it should be disregarded and to apologise. We take this type of incident very seriously indeed and so a thorough investigation, overseen by the headmaster Tony Little and led by the tutor for admissions, is being carried out to find out exactly what went wrong and ensure it cannot happen again. Eton College offers its sincere apologies to those boys concerned and their families. We deeply regret the confusion and upset this must have caused."\nquestion: Fo

## Load URLs and web pages

In [ ]:
import os
os.environ.setdefault("USER_AGENT", "CAM_DS_C401_RAG/1.0")

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://fourthrev.com/blog-announcing-data-science-career-accelerator/")


/tmp/ipykernel_4547/2394449727.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [ ]:
docs = loader.load()

In [ ]:
docs

[Document(metadata={'source': 'https://fourthrev.com/blog-announcing-data-science-career-accelerator/', 'title': 'Announcing: New Data Science Career Accelerator | FourthRev', 'description': "We're launching our new Data Science Career Accelerator in collaboration with the University of Cambridge Professional and Continuing Education (PACE). Read more here.", 'language': 'en-US'}, page_content="\n\n\n\n\n\n\n\nAnnouncing: New Data Science Career Accelerator | FourthRev\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\nCareer Accelerators\n\nKing’s Product Management\nKing’s  UX & UI Product Design\nLSE Digital Marketing\nLSE Data Analytics\nLSE AI Leadership\nCambridge PACE Data Science\nCambridge PACE Cybersecurity\nMonash AI Strategy and Leadership\nMonash Data Analytics\n\n\nWhy FourthRev\n\nAbout Us\nHow it work

### Load the PDF files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://arxiv.org/pdf/1706.03762")


In [ ]:
docs = loader.load()


In [ ]:
docs[4].page_content

'output values. These are concatenated and once again projected, resulting in the final values, as\ndepicted in Figure 2.\nMulti-head attention allows the model to jointly attend to information from different representation\nsubspaces at different positions. With a single attention head, averaging inhibits this.\nMultiHead(Q,K,V ) = Concat(head 1,..., headh)W O\nwhere headi = Attention(QW Q\ni ,KW K\ni ,VW V\ni )\nWhere the projections are parameter matricesW Q\ni ∈ Rdmodel×dk,W K\ni ∈ Rdmodel×dk,W V\ni ∈ Rdmodel×dv\nandW O∈ Rhdv×dmodel.\nIn this work we employ h = 8 parallel attention layers, or heads. For each of these we use\ndk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost\nis similar to that of single-head attention with full dimensionality.\n3.2.3 Applications of Attention in our Model\nThe Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decode

## Split documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=30)

chunked_docs = splitter.split_documents(docs)

In [ ]:
chunked_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1706.03762', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗'),
 

## Create the embeddings and the retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Set up vector databases

## FAISS vector database

In [ ]:
db = FAISS.from_documents(chunked_docs, embedding_function )

## Chroma vector database

In [ ]:
from langchain_chroma import Chroma
db = Chroma.from_documents(chunked_docs, embedding_function)

We need a way to return (retrieve) the documents given an unstructured query. For that, we will use the `as_retriever` method, using the `db` as a backbone:
- `search_type="similarity"` means we want to perform similarity search between the query and documents.
- `search_kwargs={'k': 4}` instructs the retriever to return top 4 results.


In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 4}
)

The vector database and retriever are now set up.

### Save the vector database

In [ ]:
!mkdir '/content/docs'

mkdir: cannot create directory ‘/content/docs’: File exists


In [ ]:
persist_directory = '/content/docs'

In [ ]:
!rm -rf /content/docs  # remove old database files if any

In [ ]:
db = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embedding_function,
    persist_directory=persist_directory
)

In [ ]:
print(db._collection.count())

90


## Load the quantised model

For this example, we chose [`HuggingFaceH4/zephyr-7b-beta`](https://huggingface.co/HuggingFaceH4/zephyr-7b-beta), a small but powerful model.

With many models being released every week, you may want to substitute this model with the latest one. The best way to keep track of open source LLMs is to check the [open-source LLM leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard).

To make inference faster, we will load the quantised version of the model:

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("The local Zephyr 4-bit generation section requires a CUDA GPU runtime. In Colab, select Runtime > Change runtime type > GPU before running this section.")

model_name = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## Set up the LLM chain

Finally, we have all the pieces we need to set up the LLM chain.

First, create a text_generation pipeline using the loaded model and its tokeniser.

Next, create a prompt template. This should follow the format of the model, so if you substitute the model checkpoint, ensure that you use the appropriate formatting.

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser

model.generation_config.max_length = None
model.generation_config.max_new_tokens = 400
model.generation_config.temperature = 0.2
model.generation_config.do_sample = True
model.generation_config.repetition_penalty = 1.1

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    return_full_text=True,
    clean_up_tokenization_spaces=False,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>

 """

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()


Finally, we need to combine the `llm_chain` with the retriever to create a RAG chain. We pass the original question through to the final generation step, as well as the retrieved context docs:

In [ ]:
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()

def format_docs(documents):
    return "\n\n".join(document.page_content for document in documents)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_chain
)


## Compare the results

Let's see the difference RAG makes in generating answers to the library-specific questions.

In [ ]:
question = "What is self attention according to the paper?"

First, let's see what kind of answer we can get with just the model itself, with no context added:

In [ ]:
llm_chain.invoke({"context":"", "question": question})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n\n\n</s>\n<|user|>\nWhat is self attention according to the paper?\n</s>\n<|assistant|>\n\n Self attention, as introduced in the paper "Attention Is All You Need" by Vaswani et al., refers to a mechanism that allows a neural network to attend to specific parts of an input sequence without the need for external guidance or supervision. In other words, it enables the model to learn which elements are most important and relevant to the task at hand through internal computations, rather than relying on predefined rules or explicit signals. This technique has shown significant improvements in performance on various natural language processing tasks, such as machine translation and text generation, over traditional approaches that rely on external attention mechanisms.'

In [ ]:
rag_chain.invoke(question)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].\n\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nFigure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the\nsentence. We give two such examples above, from two different heads from the encoder self-attention\nat layer 5 of 6. The heads clearly learned to perform different tasks.\n15\n\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nFigure 3: An example of the attention mechanism following long-distance depen

## Use OpenAI models instead

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except userdata.SecretNotFoundError:
    raise RuntimeError("Add an OPENAI_API_KEY secret in Colab and enable notebook access before running the OpenAI-backed RAG cells.")


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    reasoning_effort="low",
    use_responses_api=True,
)


In [ ]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()

In [ ]:
question = "What is self attention according to the paper?"

In [ ]:
llm_chain.invoke({"context":"", "question": question})

'Self-attention is an attention mechanism in which each element of a sequence attends to—and incorporates information from—other elements within the same sequence, including itself. The model creates a query, key, and value representation for each token, computes attention weights from query–key similarities, and uses those weights to form a weighted sum of the value representations. This allows each token’s representation to depend on the entire input sequence.'

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_chain
)


In [ ]:
rag_chain.invoke(question)

'According to the paper, **self-attention (or intra-attention)** is an attention mechanism that relates different positions within a single sequence to compute a representation of that sequence. It allows each word or position to attend to other words in the same sequence, helping the model capture relationships such as long-distance dependencies and anaphora.'

## Key information
You have learned how to perform retrieval augmented generation (RAG) with both an open-source model and a closed-source model.

## Reflect
Compare the outputs from two or more models, and note your findings and observations.

> Select the pen from the toolbar to add your entry.